# SinhalaMMLU: Qwen3.5-4B zero-shot baseline
Set **Runtime → Change runtime type → GPU**. Accept the dataset terms at [hf.co/datasets/naist-nlp/SinhalaMMLU](https://huggingface.co/datasets/naist-nlp/SinhalaMMLU) and add your Hugging Face token as a Colab secret named `HF_TOKEN`.

In [ ]:
%pip install -q "transformers==5.19.0" accelerate

## Model and official prompt
The prompt strings are copied verbatim from the official `src/util_prompt.py`. The prediction is the option digit with the highest next-token probability, as in the official evaluation.

In [ ]:
from pathlib import Path
import json

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3.5-4B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda")  # text-only: vision tower not loaded
DIGIT_IDS = tokenizer.convert_tokens_to_ids(list("12345"))

SYSTEM = "You must output the Answer"
INTRO = "මෙය {} විෂයයට අදාළ බහුවරණ ප්‍රශ්නයකි.\n"
INSTRUCTION = {
    "easy": "පහත ප්‍රශ්නයට 1, 2, 3, 4 යන පිළිතුරුවලින් නිවැරදි හෝ ඉතාමත් ගැළපෙන පිළිතුර තෝරන්න.\n\n",
    "medium": "පහත ප්‍රශ්නයට 1, 2, 3, 4  යන පිළිතුරුවලින් නිවැරදි හෝ ඉතාමත් ගැළපෙන පිළිතුර තෝරාගන්න.\n\n",
    "hard": "පහත ප්‍රශ්නයට 1, 2, 3, 4, 5 යන පිළිතුරුවලින් නිවැරදි හෝ ඉතාමත් ගැළපෙන පිළිතුර තෝරාගන්න.\n\n",
}


def build_prompt(subject: str, level: str, question: str, choices: list[str]) -> str:
    options = "\n".join(f"{i}. {choice.strip()}" for i, choice in enumerate(choices, start=1))
    return f"{INTRO.format(subject)}{INSTRUCTION[level]}ප්‍රශ්නය: {question}\n{options}\nපිළිතුර:"


@torch.inference_mode()
def option_probs(subject: str, level: str, question: str, choices: list[str]) -> torch.Tensor:
    """Probability of each option digit as the model's next token (index 0 = option 1)."""
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": build_prompt(subject, level, question, choices)}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_dict=True, return_tensors="pt",
        enable_thinking=False,  # Qwen3.5 thinks by default; the next token would then be reasoning, not a digit
    ).to(model.device)
    return model(**inputs, logits_to_keep=1).logits[0, -1, DIGIT_IDS[: len(choices)]].float().softmax(-1)

## Try your own questions
Edit the list and rerun this cell. `answer` is the correct option number (from 1). `level` only changes the instruction line; use `"hard"` for 5 options.

In [ ]:
MY_QUESTIONS = [
    {"subject": "භූගෝල විද්‍යාව", "level": "easy", "question": "ශ්‍රී ලංකාවේ දිගම ගඟ කුමක් ද?",
     "choices": ["කැලණි ගඟ", "කළු ගඟ", "මහවැලි ගඟ", "වළවේ ගඟ"], "answer": 3},
    {"subject": "විද්‍යාව", "level": "easy", "question": "ජලයේ රසායනික සූත්‍රය කුමක් ද?",
     "choices": ["H2O", "CO2", "O2", "NaCl"], "answer": 1},
    {"subject": "ඉතිහාසය", "level": "hard", "question": "මිහිඳු මහරහතන් වහන්සේ ලංකාවට වැඩම කළේ කුමන රජු සමයේ ද?",
     "choices": ["පණ්ඩුකාභය", "දුටුගැමුණු", "වළගම්බා", "දේවානම්පියතිස්ස", "මහසෙන්"], "answer": 4},
]

for q in MY_QUESTIONS:
    probs = option_probs(q["subject"], q["level"], q["question"], q["choices"])
    pred = int(probs.argmax()) + 1
    print(f"{'✓' if pred == q['answer'] else '✗'} predicted {pred}, correct {q['answer']} | {q['question']}")
    print("   " + " | ".join(f"{i}. {c}: {p:.0%}" for i, (c, p) in enumerate(zip(q["choices"], probs.tolist()), start=1)))

## Score the dataset
The public split has Easy questions only. For all three levels, point `DATA_DIR` at the dev set: a folder with `easy/`, `medium/` and `hard/` subfolders of `.json` files that include gold answers. The macro score (the average over levels) is the ranking metric.

In [ ]:
from huggingface_hub import snapshot_download

DATA_DIR = snapshot_download("naist-nlp/SinhalaMMLU", repo_type="dataset") + "/TEST"

accuracy = {}
for level in INSTRUCTION:  # the level comes from the folder name, as in the official runner
    items = [q for path in sorted(Path(DATA_DIR, level).glob("*.json")) for q in json.loads(path.read_text(encoding="utf-8"))]
    if items:
        hits = [int(option_probs(q["metadata"]["subject_original"], level, q["question"], q["choices"]).argmax()) + 1 == int(q["answer"])
                for q in tqdm(items, desc=level)]
        accuracy[level] = sum(hits) / len(hits)
        print(f"{level}: {accuracy[level]:.2%} (n={len(hits)})")
print(f"macro: {sum(accuracy.values()) / len(accuracy):.2%} over {len(accuracy)} of 3 levels")